# 3.6 softmax 回归的从零开始实现

本节基于 Fashion-MNIST 服装分类数据集，从零完整实现 softmax 回归分类算法，覆盖参数初始化、softmax 运算、交叉熵损失、精度计算与完整训练流程，仅使用张量与自动求导完成全部逻辑，深入理解分类模型的底层原理。

In [ ]:
import torch
from IPython import display
from d2l import torch as d2l

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

## 3.6.1 初始化模型参数
将每张28×28的灰度图像展平为长度784的一维特征向量；数据集共10个类别，因此模型输出维度为10。
- 权重矩阵 $W$ 形状：(784, 10)，从均值为0、标准差0.01的正态分布随机初始化
- 偏置向量 $b$ 形状：(10,)，初始化为0


In [ ]:
num_inputs = 784
num_outputs = 10

W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)

## 3.6.2 定义softmax操作
softmax将任意实数值输出转换为合法的概率分布：每行代表一个样本在所有类别上的概率，所有值非负且每行总和为1。
运算分为三步：
1.	对每个输出值取指数
2.	按行求和得到每个样本的归一化常数（配分函数）
3.	每行除以对应归一化常数

公式： $$\text{softmax}(X_{ij}) = \frac{\exp(X_{ij})}{\sum_{k} \exp(X_{ik})}$$

先演示张量按轴求和的特性，keepdim=True保留维度，便于后续广播运算。


In [ ]:
X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
X.sum(0, keepdim=True), X.sum(1, keepdim=True)

实现softmax函数，利用广播机制完成逐行归一化。

In [ ]:
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition  # 这里应用了广播机制

验证softmax输出特性：每行概率之和为1，所有值非负。

In [ ]:
X = torch.normal(0, 1, (2, 5))
X_prob = softmax(X)
X_prob, X_prob.sum(1)

## 3.6.3 定义模型
前向传播流程：先将批量图像张量展平为特征向量，经过线性变换得到输出logits，再通过softmax转换为类别概率分布。


In [ ]:
def net(X):
    return softmax(torch.matmul(X.reshape((-1, W.shape[0])), W) + b)

## 3.6.4 定义交叉熵损失函数
交叉熵是分类任务最常用的损失函数，单样本损失取值为真实类别对应预测概率的负对数： $$l(y,\hat{y}) = -\log \hat{y}_y$$ 其中 $\hat{y}_y$ 表示真实类别$y$对应的预测概率。

利用PyTorch高级索引批量取出每个样本真实类别的预测概率，无需循环，高效计算批量损失。

先演示索引取值逻辑：


In [ ]:
y = torch.tensor([0, 2])
y_hat = torch.tensor([[0.1, 0.3, 0.6], [0.3, 0.2, 0.5]])
y_hat[[0, 1], y]

实现交叉熵损失函数，返回每个样本的损失值。

In [ ]:
def cross_entropy(y_hat, y):
    return - torch.log(y_hat[range(len(y_hat)), y])

cross_entropy(y_hat, y)

## 3.6.5 分类精度计算
分类精度 = 正确预测样本数 / 总样本数，是分类任务最核心的性能评估指标。

##### 1. 批量精度计算
通过argmax取每行最大概率的类别索引，与真实标签对比，统计正确预测的样本数量。


In [ ]:
def accuracy(y_hat, y):  #@save
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

验证精度计算结果：

In [ ]:
accuracy(y_hat, y) / len(y)

##### 2. 累加器工具类

用于遍历数据集时，累加正确预测数与总样本数等多个统计量。

In [ ]:
class Accumulator:  #@save
    """在n个变量上累加"""
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

##### 3. 数据集整体精度评估

遍历指定数据迭代器，计算模型在完整数据集上的分类精度。

In [ ]:
def evaluate_accuracy(net, data_iter):  #@save
    """计算在指定数据集上模型的精度"""
    if isinstance(net, torch.nn.Module):
        net.eval()  # 将模型设置为评估模式
    metric = Accumulator(2)  # 正确预测数、预测总数
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

测试随机初始化模型的精度，10分类任务随机精度约为0.1。

In [ ]:
evaluate_accuracy(net, test_iter)

## 3.6.6 模型训练
重构通用训练流程，同时兼容手写优化器与PyTorch内置优化器，配套动画可视化工具实时监控训练过程。
##### 1. 单迭代周期训练函数
完成一个epoch的完整训练：
- 切换模型为训练模式
- 逐批量执行前向传播、损失计算、反向传播、参数更新
- 累加统计训练损失、训练精度与总样本数
- 自动适配内置优化器与自定义优化器两种场景


In [ ]:
def train_epoch_ch3(net, train_iter, loss, updater):  #@save
    """训练模型一个迭代周期（定义见第3章）"""
    # 将模型设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()
    # 训练损失总和、训练准确度总和、样本数
    metric = Accumulator(3)
    for X, y in train_iter:
        # 计算梯度并更新参数
        y_hat = net(X)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            # 使用PyTorch内置的优化器和损失函数
            updater.zero_grad()
            l.mean().backward()
            updater.step()
        else:
            # 使用定制的优化器和损失函数
            l.sum().backward()
            updater(X.shape[0])
        metric.add(float(l.sum()), accuracy(y_hat, y), y.numel())
    # 返回训练损失和训练精度
    return metric[0] / metric[2], metric[1] / metric[2]

##### 2. 动画绘图工具类

在训练过程中实时绘制损失与精度曲线，动态展示训练收敛过程。

In [ ]:
class Animator:  #@save
    """在动画中绘制数据"""
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 fmts=('-', 'm--', 'g-.', 'r:'), nrows=1, ncols=1,
                 figsize=(3.5, 2.5)):
        # 增量地绘制多条线
        if legend is None:
            legend = []
        d2l.use_svg_display()
        self.fig, self.axes = d2l.plt.subplots(nrows, ncols, figsize=figsize)
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        # 使用lambda函数捕获参数
        self.config_axes = lambda: d2l.set_axes(
            self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend)
        self.X, self.Y, self.fmts = None, None, fmts

    def add(self, x, y):
        # 向图表中添加多个数据点
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x, y, fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)

##### 3. 完整训练函数

循环执行多个迭代周期，每个周期结束后在测试集上评估精度，通过动画实时展示训练损失、训练精度与测试精度三条曲线。

In [ ]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):  #@save
    """训练模型（定义见第3章）"""
    animator = Animator(xlabel='epoch', xlim=[1, num_epochs], ylim=[0.3, 0.9],
                        legend=['train loss', 'train acc', 'test acc'])
    for epoch in range(num_epochs):
        train_metrics = train_epoch_ch3(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
        animator.add(epoch + 1, train_metrics + (test_acc,))
    train_loss, train_acc = train_metrics
    assert train_loss < 0.5, train_loss
    assert train_acc <= 1 and train_acc > 0.7, train_acc
    assert test_acc <= 1 and test_acc > 0.7, test_acc

##### 4. 定义优化器并启动训练

使用 3.2 节实现的小批量随机梯度下降作为优化器，设置学习率 0.1，训练 10 个迭代周期。

In [ ]:
lr = 0.1

def updater(batch_size):
    return d2l.sgd([W, b], lr, batch_size)
num_epochs = 10
train_ch3(net, train_iter, test_iter, cross_entropy, num_epochs, updater)

#### 3.6.7 预测可视化

从测试集中取出一批样本，对比真实标签与模型预测结果，直观验证分类效果。

In [ ]:
def predict_ch3(net, test_iter, n=6):  #@save
    """预测标签（定义见第3章）"""
    for X, y in test_iter:
        break
    trues = d2l.get_fashion_mnist_labels(y)
    preds = d2l.get_fashion_mnist_labels(net(X).argmax(axis=1))
    titles = [true +'\n' + pred for true, pred in zip(trues, preds)]
    d2l.show_images(
        X[0:n].reshape((n, 28, 28)), 1, n, titles=titles[0:n])

predict_ch3(net, test_iter)